In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder,StandardScaler,MinMaxScaler

In [40]:
df=pd.read_csv('titanic_data_updated (1).csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


splitting data

In [41]:
df.drop(['PassengerId','Name','Ticket'], axis=1, inplace=True)

#family_size creation
df['Family_Size']=df['SibSp']+df['Parch']+1

#feature and target extract
X=df.drop('Survived', axis=1)
y=df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

imputation of data

In [42]:
imputer_transformer=ColumnTransformer(transformers=[
    ('age', SimpleImputer(missing_values=np.nan, strategy='mean'), ['Age']),
     ('embarked', SimpleImputer(missing_values=np.nan, strategy='most_frequent'), ['Embarked']),
      ('cabin', SimpleImputer(missing_values=np.nan, strategy='constant',fill_value='Missing',add_indicator=True), ['Cabin'])
    
],
  remainder='passthrough',
 verbose_feature_names_out=False
)
imputer_transformer.set_output(transform='pandas')
imputer_transformer.fit(X_train)
X_train_imputed=imputer_transformer.transform(X_train)
X_test_imputed=imputer_transformer.transform(X_test)


outlier handling of age

In [43]:
mean_of_age=X_train['Age'].mean()
std_of_age=X_train['Age'].std()

X_train['zscore_age']=(X_train['Age']-mean_of_age)/std_of_age

X_train=X_train[abs(X_train['zscore_age']) < 3]
X_train.drop(['zscore_age'], axis=1, inplace=True)

outlier handling of Fare

In [44]:
fare_Q1=X_train['Fare'].quantile(0.25)
fare_Q3=X_train['Fare'].quantile(0.75)
fare_IQR=fare_Q3-fare_Q1
fare_minimum=max(0,fare_Q1-1.5*fare_IQR)
fare_maximum=fare_Q3+1.5*fare_IQR
X_train['Fare']=X_train['Fare'].clip(fare_minimum,fare_maximum)

encoding and scaling

In [46]:
encoder_scaler = ColumnTransformer(
    transformers=[
       
        ('pclass', OrdinalEncoder(categories=[['third', 'second', 'first']]), ['Pclass']),
        ('embarked_sex', OneHotEncoder(sparse_output=False, drop='first'), ['Embarked', 'Sex', 'Cabin_Deck']),
        ('age_scaler', StandardScaler(), ['Age']),
        ('fare_scaler', StandardScaler(), ['Fare', 'Family_Size'])
    ], 
    remainder='passthrough', 
    verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas')


encoder_scaler.fit(X_train)

X_train.drop(['Cabin', 'SibSp', 'Parch'], axis=1, inplace=True)
X_test.drop(['Cabin', 'SibSp', 'Parch'], axis=1, inplace=True)

In [47]:
X_train

,Pclass,Sex,Age,Fare,Embarked,Family_Size,Cabin_Deck
331,first,male,45.5,28.5000,S,1,C
733,second,male,23.0,13.0000,S,1,n
382,third,male,32.0,7.9250,S,1,n
704,third,male,26.0,7.8542,S,2,n
813,third,female,6.0,31.2750,S,7,n
...,...,...,...,...,...,...,...
71,third,female,16.0,46.9000,S,8,n
106,third,female,21.0,7.6500,S,1,n
860,third,male,41.0,14.1083,S,3,n
435,first,female,14.0,72.6515,S,4,B
